In [1]:
import pandas as pd
import numpy as np
import requests
import os
import json

## Flightaware AeroAPI

In [2]:
with open('../apiKeys/apiKeys.json') as f:
    API_KEYS = json.load(f)
    f.close()

In [3]:
AEROAPI_BASE_URL = "https://aeroapi.flightaware.com/aeroapi"

def get_operator_flights(operator_id: str) -> dict:
    url = f"{AEROAPI_BASE_URL}/operators/{operator_id}/flights"
    headers = {"x-apikey": API_KEYS['FA_API_KEY']}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

# Example usage:
data = pd.DataFrame()
for airline in ["UAL", "AAL", "DL", "WN", "AS", "B6"]:
    flights = get_operator_flights(airline)
    flights_df = pd.DataFrame(flights["scheduled"])
    flights_df.to_csv(f"data/{airline}_flights.csv")
    data = pd.concat([data, flights_df]).reset_index(drop=True)
data

/var/folders/mh/b5m2p8fd04x2knz5_hwd1rqw0000gn/T/ipykernel_67081/1058542627.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, flights_df]).reset_index(drop=True)


,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,route,baggage_claim,seats_cabin_business,seats_cabin_coach,seats_cabin_first,gate_origin,gate_destination,terminal_origin,terminal_destination,type
0,UAL4206,UAL4206,UA4206,None,None,UAL4206-1781985842-fa-318p,UAL,UAL,UA,4206,...,None,None,NaN,NaN,20.0,None,46,None,None,Airline
1,UAL2612,UAL2612,UA2612,None,None,UAL2612-1781974174-fa-2403p,UAL,UAL,UA,2612,...,OTT020015 FUBRR PHLBO4,None,NaN,NaN,12.0,B11,A28,None,A,Airline
2,UAL1611,UAL1611,UA1611,None,None,UAL1611-1782016446-fa-174p,UAL,UAL,UA,1611,...,OTT047031 FUBRR PHLBO4,None,NaN,NaN,20.0,C4,A21,1,A,Airline
3,UAL1156,UAL1156,UA1156,None,None,UAL1156-1781913800-fa-5756p,UAL,UAL,UA,1156,...,COATE Q436 KAYYS WYNDE3,None,NaN,NaN,16.0,46,E17,B,2,Airline
4,UAL2644,UAL2644,UA2644,None,None,UAL2644-1781908072-fa-1624p,UAL,UAL,UA,2644,...,MXE PENSY J110 AIR APE J178 FWA JOT J60 LNK OB...,None,NaN,NaN,NaN,None,None,None,None,Airline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,JBU474,JBU474,B6474,None,None,JBU474-1781940044-airline-523p,JBU,JBU,B6,474,...,SCAPA1F SCAPA L325 JOSHE PANMO L458 THANK L456...,None,NaN,NaN,NaN,6,C20,None,C,Airline
86,JBU2134,JBU2134,B62134,None,None,JBU2134-1781940044-airline-261p,JBU,JBU,B6,2134,...,ACONY3 SAPPO Y280 SUMAC ZFP CCLUB MUNGI1,None,NaN,NaN,NaN,A8,232,A,C,Airline
87,JBU137,JBU137,B6137,None,None,JBU137-1781932435-airline-1691p,JBU,JBU,B6,137,...,THUMB1 CCC SPDEY Y488 HERCC Y488 STERN Y493 TU...,None,NaN,NaN,NaN,5,234,None,C,Airline
88,JBU438,JBU438,B6438,None,None,JBU438-1781940044-airline-509p,JBU,JBU,B6,438,...,FATHE4 VIYAP Q87 JROSS ELLDE Q97 PAACK JAMIE C...,None,NaN,NaN,NaN,253A,5,C,None,Airline


## Aircraft Data

In [10]:
with open('aircraft_data/UAL_aircraft_data.json') as f:
    raw = json.load(f)

def _flatten_ual_data(data: dict) -> pd.DataFrame:
    rows = []

    def _process(aircraft_name: str, d: dict):
        for key, val in d.items():
            if not isinstance(val, dict):
                continue
            if any(isinstance(v, dict) for v in val.values()):
                # malformed nesting: key is actually another aircraft variant
                _process(key, val)
            else:
                rows.append({"aircraft": aircraft_name, "cabin_class": key, **val})

    for aircraft, cabins in data.items():
        _process(aircraft, cabins)

    return pd.DataFrame(rows)

ual_aircraft_df = _flatten_ual_data(raw)
ual_aircraft_df

,aircraft,cabin_class,Number of seats,Seat numbers,Exit rows/doors,Seat configuration,Standard seat pitch,Standard seat recline,Limited/Zero seat recline,Seat width,Movable aisle armrests,Fixed bassinet,Entertainment,Wi-Fi,Power outlets,USB ports,Fixed bassinets,Wireless charging,Bulk head seats,Over wing rows
0,B777-200,United First®,28,1A-4L,Front of cabin,2–4–2,"6'4"" (193 cm) sleeping space",180°,N/A,"19"" (48.3 cm)",All rows,No,Seatback entertainment and personal device ent...,True,True,False,NaN,NaN,NaN,NaN
1,B777-200,United Economy Plus®,102,"17A-24L; ABCJKL in rows 25, 26, 39; 40D-G","Rows 17, 39",3-4-3,"35"" (88 cm)","4"" (10 cm)",N/A,"17.1"" (43.4 cm)",All rows,40FG,Personal device entertainment,True,True,True,NaN,NaN,NaN,NaN
2,B777-200,United Economy®,234,"D-G in rows 25, 26; 27A-37L; 40ABCJKL; 41A-53G",Back of cabin,3-4-3,"31"" (78 cm)","3"" (7 cm)",N/A,"17.1"" (43.4 cm)",All rows,No,Personal device entertainment,True,True,False,NaN,NaN,NaN,NaN
3,B777-200ER Version 1,Polaris® Business Class,50,1A-15L,Front of cabin and between rows 8 and 9,1–2–1,"6'6"" (198 cm) sleeping space",180°,N/A,"22"" (55.9 cm)","Rows 2, 4, 6, 8, 10, 12",NaN,Seatback entertainment and personal device ent...,Yes,Yes,Yes,"9A, 9L",NaN,NaN,NaN
4,B777-200ER Version 1,Premium Plus®,24,20A–22L,No,2–4–2,"38"" (96.5 cm)","6"" (15 cm)",N/A,"18.5"" (47 cm)",All aisle seats,NaN,Seatback entertainment and personal device ent...,Yes,Yes,Yes,20DEFG,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,Embraer E175 Version 1,United Economy Plus®,32,7A-16D,No,2-2,"34"" (86 cm)","3"" (7 cm)",N/A,"18.2"" (46 cm)",Rows 8-16,NaN,Personal device entertainment,True,True,False,No,NaN,NaN,09-12
101,Embraer E175 Version 1,United Economy®,26,17A-23B,Back of cabin,2-2,"31"" (78 cm)","2"" (5 cm)",N/A,"18.2"" (46 cm)",All rows,NaN,Personal device entertainment,True,True,False,No,NaN,NaN,No
102,Embraer E175 Version 2,United First®,12,1A-4D,Front of cabin,1-2,"37"" (93 cm)","5"" (12 cm)",N/A,"20"" (50 cm)",Rows 2-4,NaN,Personal device entertainment,True,True,False,No,NaN,1,No
103,Embraer E175 Version 2,United Economy Plus®,16,7A-10D,No,2-2,"34"" (86 cm)","3"" (7 cm)",N/A,"18"" (45 cm)",Rows 8-10,NaN,Personal device entertainment,True,False,False,No,NaN,No,No


In [11]:
data[data['operator_icao'] == 'UAL']

,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,route,baggage_claim,seats_cabin_business,seats_cabin_coach,seats_cabin_first,gate_origin,gate_destination,terminal_origin,terminal_destination,type
0,UAL4206,UAL4206,UA4206,None,None,UAL4206-1781985842-fa-318p,UAL,UAL,UA,4206,...,None,None,NaN,NaN,20.0,None,46,None,None,Airline
1,UAL2612,UAL2612,UA2612,None,None,UAL2612-1781974174-fa-2403p,UAL,UAL,UA,2612,...,OTT020015 FUBRR PHLBO4,None,NaN,NaN,12.0,B11,A28,None,A,Airline
2,UAL1611,UAL1611,UA1611,None,None,UAL1611-1782016446-fa-174p,UAL,UAL,UA,1611,...,OTT047031 FUBRR PHLBO4,None,NaN,NaN,20.0,C4,A21,1,A,Airline
3,UAL1156,UAL1156,UA1156,None,None,UAL1156-1781913800-fa-5756p,UAL,UAL,UA,1156,...,COATE Q436 KAYYS WYNDE3,None,NaN,NaN,16.0,46,E17,B,2,Airline
4,UAL2644,UAL2644,UA2644,None,None,UAL2644-1781908072-fa-1624p,UAL,UAL,UA,2644,...,MXE PENSY J110 AIR APE J178 FWA JOT J60 LNK OB...,None,NaN,NaN,NaN,None,None,None,None,Airline
5,UAL498,UAL498,UA498,None,None,UAL498-1781913879-fa-4p,UAL,UAL,UA,498,...,CHLDR5 ANSWA TXK BYP UKW HNKER CNX GNDLA CRUDD...,None,NaN,NaN,NaN,None,None,None,None,Airline
6,UAL3919,UAL3919,UA3919,None,None,UAL3919-1782137318-fa-546p,UAL,UAL,UA,3919,...,ALPIN5 KICNE DDY ONL FOD MYRRS FYTTE7,None,NaN,NaN,12.0,None,NR31,None,None,Airline
7,UAL1848,UAL1848,UA1848,None,None,UAL1848-1781913935-fa-1320p,UAL,UAL,UA,1848,...,+ELVAE NECCK WHITE Q409 MRPIT CEELY Q172 YUTEE...,None,NaN,NaN,12.0,A17,T21,A,N,Airline
8,UAL2857,UAL2857,UA2857,None,None,UAL2857-1781913838-fa-767p,UAL,UAL,UA,2857,...,JCOBY4 SCOOB GUILD Q409 SESUE PANDY TWINS NOKI...,None,NaN,NaN,20.0,C17,C2,None,C,Airline
9,UAL1199,UAL1199,UA1199,None,None,UAL1199-1781910915-fa-767p,UAL,UAL,UA,1199,...,RIDDG2 SKOTT TOFUU FLATI5,None,NaN,NaN,16.0,2,B44,None,None,Airline


## Flight Capacity (Available Seat Miles)

Join the United seat-map data onto each scheduled flight to compute **Available Seat Miles (ASM)** per cabin — `seats × route_distance`. ASM is the foundational airline capacity metric and the unit we'll later multiply by fares / load factors to model revenue.

The bridge between the two datasets is a mapping from FlightAware `aircraft_type` codes (`B77W`, `B738`, …) to United's marketing seat-map names (`B777-300ER`, `737-800 Version 1`, …).

In [12]:
import ast

# FlightAware `aircraft_type` code -> United seat-map name (the `aircraft` key in ual_aircraft_df).
# Where United operates multiple configurations of a type, we default to "Version 1".
AIRCRAFT_TYPE_MAP = {
    # ── Wide-bodies ────────────────────────────────────────────────────────
    "B77W": "B777-300ER",              # 777-300ER
    "B772": "B777-200",               # 777-200
    "B77L": "B777-200",               # 777-200LR
    "B788": "B787-8",                 # 787-8 Dreamliner
    "B789": "B787-9 Version 1",       # 787-9
    "B78X": "B787-10",                # 787-10
    "B763": "767-300ER Version 1",    # 767-300ER
    "B764": "767-400ER",              # 767-400ER
    "B752": "757-200",                # 757-200
    "B753": "757-300",                # 757-300
    # ── Narrow-bodies ──────────────────────────────────────────────────────
    "B737": "737-700",                # 737-700
    "B738": "737-800 Version 1",      # 737-800
    "B739": "737-900 Version 1",      # 737-900 / 900ER
    "B38M": "737 MAX 8 Version 1",    # 737 MAX 8
    "B39M": "737 MAX 9 Version 1",    # 737 MAX 9
    "A319": "A319",
    "A320": "A320",
    "A21N": "A321neo",                # A321neo / XLR
    # ── Regional jets ──────────────────────────────────────────────────────
    "CRJ2": "CRJ200",
    "CL65": "CRJ550",
    "CRJ7": "CRJ700",
    "E170": "Embraer E170",
    "E75L": "Embraer E175 Version 1",
    "E75S": "Embraer E175 Version 1",
}


def _iata(cell):
    """Pull the IATA code out of FlightAware's stringified origin/destination dict."""
    try:
        return ast.literal_eval(cell).get("code_iata")
    except (ValueError, SyntaxError, AttributeError, TypeError):
        return None


def compute_flight_capacity(flights_df, aircraft_df, type_map=AIRCRAFT_TYPE_MAP):
    """Join seat-map config onto each flight and compute Available Seat Miles (ASM) per cabin.

    Returns:
        capacity_df: one row per (flight, cabin_class) with seats and ASM = seats * route_distance.
        unmapped:    set of aircraft_type codes present in flights but missing from `type_map`.
    """
    seats = (aircraft_df[["aircraft", "cabin_class", "Number of seats"]]
             .rename(columns={"Number of seats": "seats"})
             .copy())
    seats["seats"] = pd.to_numeric(seats["seats"], errors="coerce")

    f = flights_df.copy()
    f["seatmap_name"] = f["aircraft_type"].map(type_map)

    have_seatmap = set(seats["aircraft"].unique())
    unmapped = set(
        f.loc[~f["seatmap_name"].isin(have_seatmap), "aircraft_type"].dropna().unique()
    )

    id_col = "fa_flight_id" if "fa_flight_id" in f.columns else "ident"
    f = f.rename(columns={id_col: "flight_id"})
    f["origin_iata"] = f["origin"].map(_iata)
    f["destination_iata"] = f["destination"].map(_iata)

    merged = f.merge(seats, left_on="seatmap_name", right_on="aircraft", how="inner")
    merged["ASM"] = merged["seats"] * merged["route_distance"]

    cols = ["flight_id", "ident", "aircraft_type", "seatmap_name",
            "origin_iata", "destination_iata", "route_distance",
            "cabin_class", "seats", "ASM"]
    cols = [c for c in cols if c in merged.columns]
    return merged[cols].reset_index(drop=True), unmapped


In [13]:
# Load UAL flights from the saved CSV (so this runs without a live API call) and compute capacity.
ual_flights = pd.read_csv("data/UAL_flights.csv")

capacity_df, unmapped = compute_flight_capacity(ual_flights, ual_aircraft_df)
print(f"{capacity_df['flight_id'].nunique()} flights joined, "
      f"{len(capacity_df)} (flight x cabin) rows")
if unmapped:
    print(f"Unmapped aircraft_type codes (add to AIRCRAFT_TYPE_MAP): {sorted(unmapped)}")
capacity_df

15 flights joined, 45 (flight x cabin) rows


,flight_id,ident,aircraft_type,seatmap_name,origin_iata,destination_iata,route_distance,cabin_class,seats,ASM
0,UAL4206-1781985842-fa-318p,UAL4206,B739,737-900 Version 1,MCO,MCO,0,United First®,20,0
1,UAL4206-1781985842-fa-318p,UAL4206,B739,737-900 Version 1,MCO,MCO,0,United Economy Plus®,42,0
2,UAL4206-1781985842-fa-318p,UAL4206,B739,737-900 Version 1,MCO,MCO,0,United Economy®,117,0
3,UAL2612-1781974174-fa-2403p,UAL2612,B737,737-700,SRQ,EWR,1068,United Business®,12,12816
4,UAL2612-1781974174-fa-2403p,UAL2612,B737,737-700,SRQ,EWR,1068,United Economy Plus®,36,38448
5,UAL2612-1781974174-fa-2403p,UAL2612,B737,737-700,SRQ,EWR,1068,United Economy®,78,83304
6,UAL1611-1782016446-fa-174p,UAL1611,B39M,737 MAX 9 Version 1,FLL,EWR,1068,United First®,20,21360
7,UAL1611-1782016446-fa-174p,UAL1611,B39M,737 MAX 9 Version 1,FLL,EWR,1068,United Economy Plus®,48,51264
8,UAL1611-1782016446-fa-174p,UAL1611,B39M,737 MAX 9 Version 1,FLL,EWR,1068,United Economy®,111,118548
9,UAL1156-1781913800-fa-5756p,UAL1156,B738,737-800 Version 1,LGA,ORD,799,United First®,16,12784


In [14]:
# Sanity check: our joined seat totals vs the seats_cabin_* columns FlightAware already provides.
# These only overlap where FlightAware reports seats, but it's a useful cross-check on the join.
flight_seats = (capacity_df.groupby("flight_id")["seats"].sum()
                .rename("seats_joined"))

fa_seats = ual_flights.copy()
fa_seats["fa_total"] = fa_seats[["seats_cabin_first", "seats_cabin_business",
                                 "seats_cabin_coach"]].sum(axis=1, min_count=1)
fa_seats = fa_seats.set_index("fa_flight_id")["fa_total"].rename("seats_from_fa")

check = pd.concat([flight_seats, fa_seats], axis=1).dropna(subset=["seats_joined"])
check

,seats_joined,seats_from_fa
UAL1069-1781913913-fa-0p,150,NaN
UAL1156-1781913800-fa-5756p,166,16.0
UAL1199-1781910915-fa-767p,166,16.0
UAL1282-1781913877-fa-2471p,126,12.0
UAL1611-1782016446-fa-174p,179,20.0
UAL1848-1781913935-fa-1320p,126,12.0
UAL2236-1781911307-fa-822p,126,12.0
UAL2454-1781913941-fa-1940p,126,NaN
UAL2612-1781974174-fa-2403p,126,12.0
UAL2644-1781908072-fa-1624p,179,NaN


In [15]:
# Insight: ASM by aircraft and cabin, plus each aircraft's premium-vs-economy capacity mix.
PREMIUM_CABINS = ["United First®", "United Business®", "Polaris® Business Class",
                  "United Polaris® Business Class", "United Polaris® business class",
                  "Premium Plus®", "United® Premium Plus"]

asm_by_aircraft = (capacity_df
                   .groupby(["seatmap_name", "cabin_class"])["ASM"]
                   .sum()
                   .unstack(fill_value=0))

cab = capacity_df.copy()
cab["segment"] = np.where(cab["cabin_class"].isin(PREMIUM_CABINS), "premium", "economy")
mix = (cab.groupby(["seatmap_name", "segment"])["ASM"].sum()
       .unstack(fill_value=0))
mix["premium_ASM_share"] = (mix.get("premium", 0) /
                            mix.sum(axis=1)).round(3)
mix = mix.sort_values("premium_ASM_share", ascending=False)
mix

segment,economy,premium,premium_ASM_share
seatmap_name,,,
737 MAX 9 Version 1,581622,73160,0.112
737-900 Version 1,247245,31100,0.112
737-800 Version 1,232800,24832,0.096
737-700,485298,51084,0.095
A319,257412,27096,0.095
A320,40710,3540,0.080
